# RadImageNet-VQA · Ministral 3B Instruct and OpenMed MedVL

Accept access to `raidium/RadImageNet-VQA` on Hugging Face and enable the `HF_TOKEN` Kaggle secret for this notebook. Each model answers the same 500 frozen questions across five groups with real images and with 512 × 512 white images.

Enable Internet and a CUDA accelerator with at least 16 GB VRAM, then select **Save & Run All**. Allow about 20 GB of free cache space for both models plus the dataset. The setup cell installs the pinned model stack in a private environment that shares Kaggle’s GPU-enabled PyTorch. The medical model runs first, followed by the general model on the same GPU.

Both checkpoints use BF16, SDPA, native Pixtral image preprocessing, greedy decoding and the study’s original scoring rules. Both receive the same explicit empty system message so each user prompt is independent of checkpoint-specific default instructions.

Predictions are saved after each item. The final cell creates a ZIP under `/kaggle/working/`. To resume in a new session, attach the matching previous results and set `PREVIOUS_OUTPUT` to the folder containing `run_settings.json`.

In [ ]:
import atexit
import os
import site
import subprocess
import sys
import tempfile
import venv
from contextlib import ExitStack
from pathlib import Path
from IPython.core.magic import register_cell_magic
from jupyter_client import KernelManager

# Isolate the pinned model stack while reusing Kaggle's CUDA-enabled torch and torchvision.
if "ministral_runtime" in globals():
    ministral_runtime.close()
ministral_runtime = ExitStack()
atexit.register(ministral_runtime.close)
env_dir = Path(ministral_runtime.enter_context(tempfile.TemporaryDirectory(prefix="ministral-env-")))
venv.EnvBuilder(system_site_packages=True, with_pip=False).create(env_dir)
python = str(env_dir / "bin" / "python")
private_site = Path(subprocess.check_output(
    [python, "-c", "import sysconfig; print(sysconfig.get_path('purelib'))"], text=True).strip())
(private_site / "notebook-host.pth").write_text("\n".join(site.getsitepackages()) + "\n")
packages = ["transformers==5.3.0", "accelerate==1.12.0", "pillow>=11.3,<13", "datasets==4.6.1"]
subprocess.run([sys.executable, "-m", "pip", "--python", python, "install",
                "--disable-pip-version-check", *packages], check=True)
manager = KernelManager(kernel_name="python3")
manager.kernel_spec.argv = [python, "-m", "ipykernel_launcher", "-f", "{connection_file}"]
manager.start_kernel(cwd=str(Path.cwd()))
ministral_runtime.callback(manager.shutdown_kernel, now=True)
client = manager.client()
client.start_channels()
ministral_runtime.callback(client.stop_channels)
client.wait_for_ready(timeout=120)

@register_cell_magic
def ministral(line, cell):
    """Run a cell in the private kernel and stream output back to Kaggle."""
    try:
        reply = client.execute_interactive(cell, timeout=None, allow_stdin=False)["content"]
    except KeyboardInterrupt:
        manager.interrupt_kernel()
        raise
    if reply["status"] != "ok":
        raise RuntimeError(f"Ministral kernel: {reply.get('ename')}: {reply.get('evalue', '')}")

In [ ]:
%%ministral
import csv, gc, hashlib, io, json, math, os, random, re, shutil, time
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
import torch
from PIL import Image
from huggingface_hub import snapshot_download
from transformers import AutoProcessor, Mistral3ForConditionalGeneration, LogitsProcessor, LogitsProcessorList
from tempfile import TemporaryDirectory
from tqdm import tqdm
from importlib.metadata import version

MODEL_SPECS = {
    "ministral_3b_instruct": ("mistralai/Ministral-3-3B-Instruct-2512-BF16", "ef287ce8b4b5f858eaf679e9d6402265beb8624f"),
    "ministral_3b_medvl": ("OpenMed/Ministral-3B-MedVL", "4829656b3f34ae88307688a3d525ceff9daff972"),
}
RUN_MODELS = ["ministral_3b_medvl", "ministral_3b_instruct"]
PREVIOUS_OUTPUT = None  # Optional path to the prior output folder containing run_settings.json.
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
if not torch.cuda.is_available():
    raise RuntimeError("Select a Kaggle GPU accelerator and start a fresh session.")
DTYPE = torch.bfloat16  # Matches the study. Use a GPU with native BF16 support when available.
WORK = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd()
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        pass
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

from datasets import load_dataset
from sklearn.metrics import classification_report, confusion_matrix
OUTPUT_ROOT = WORK / "ministral_radimagenet_vqa_white"
CONDITIONS = ("real", "white")
MAX_NEW_TOKENS = 16
DATASET_REVISION = "9add2548cca069f088b9d0e44bfac849a53a5c31"

## Load the original five groups and frozen sample

In [ ]:
%%ministral
ds = load_dataset("raidium/RadImageNet-VQA", "benchmark", split="test",
                  revision=DATASET_REVISION, token=HF_TOKEN)
GROUPS = {   # (question_type, content_type) -> group name
    ("closed", "anatomy"):             "anatomy_yesno",
    ("closed", "pathology"):           "abnormality_yesno",   # "Does the X look abnormal?"
    ("closed", "pathology_specific"):  "pathology_yesno",
    ("multiple_choice", "anatomy"):    "anatomy_mcq",
    ("multiple_choice", "pathology"):  "pathology_mcq",
}

FROZEN_INDICES = [
    22, 82, 93, 98, 169, 210, 215, 222, 262, 273, 278, 310, 330, 331, 340, 346,
    354, 359, 379, 382, 384, 393, 411, 424, 484, 489, 494, 505, 533, 534, 539, 555,
    570, 575, 595, 597, 602, 606, 611, 631, 633, 638, 686, 687, 692, 715, 762, 778,
    795, 798, 800, 811, 825, 832, 838, 859, 861, 866, 867, 872, 901, 911, 912, 917,
    922, 925, 966, 971, 987, 1014, 1048, 1057, 1120, 1183, 1226, 1227, 1228, 1232, 1253, 1254,
    1259, 1291, 1302, 1347, 1462, 1507, 1516, 1549, 1568, 1569, 1574, 1608, 1681, 1690, 1784, 1785,
    1786, 1790, 1806, 1843, 1884, 1885, 1889, 1892, 1893, 1898, 1918, 1939, 1965, 1970, 1984, 1987,
    1992, 1997, 2095, 2098, 2117, 2118, 2123, 2146, 2182, 2283, 2287, 2299, 2346, 2352, 2357, 2371,
    2377, 2409, 2434, 2478, 2483, 2495, 2496, 2501, 2522, 2523, 2528, 2533, 2560, 2587, 2590, 2604,
    2609, 2616, 2631, 2632, 2636, 2638, 2661, 2667, 2670, 2672, 2693, 2694, 2699, 2719, 2766, 2771,
    2787, 2794, 2810, 2811, 2816, 2827, 2859, 2865, 2866, 2870, 2873, 2874, 2879, 2893, 2938, 2946,
    2951, 2954, 2955, 2959, 2960, 2977, 2994, 3028, 3037, 3094, 3116, 3117, 3120, 3122, 3235, 3246,
    3250, 3332, 3333, 3338, 3341, 3342, 3347, 3417, 3441, 3446, 3505, 3532, 3538, 3574, 3624, 3700,
    3701, 3702, 3707, 3739, 3768, 3781, 3822, 3838, 3858, 3867, 3927, 3932, 3953, 3954, 3955, 3959,
    3973, 3997, 4020, 4036, 4062, 4067, 4093, 4105, 4108, 4204, 4222, 4254, 4270, 4282, 4351, 4363,
    4366, 4396, 4440, 4445, 4470, 4493, 4494, 4499, 4522, 4552, 4588, 4619, 4620, 4625, 4672, 4691,
    4692, 4697, 4701, 4706, 4741, 4745, 4746, 4751, 4758, 4776, 4789, 4792, 4801, 4812, 4819, 4828,
    4837, 4846, 4881, 4886, 4890, 4893, 4895, 4927, 4935, 4940, 4943, 4944, 4949, 4966, 5019, 5020,
    5059, 5074, 5110, 5133, 5138, 5181, 5188, 5199, 5205, 5206, 5210, 5258, 5259, 5264, 5296, 5304,
    5305, 5309, 5313, 5318, 5411, 5412, 5416, 5417, 5431, 5439, 5444, 5451, 5479, 5556, 5561, 5591,
    5592, 5597, 5636, 5637, 5642, 5647, 5658, 5676, 5698, 5726, 5727, 5732, 5764, 5836, 5871, 5876,
    5942, 5943, 5944, 5948, 5961, 5966, 5971, 6034, 6036, 6104, 6105, 6110, 6114, 6119, 6133, 6166,
    6185, 6186, 6191, 6229, 6231, 6236, 6292, 6421, 6429, 6434, 6562, 6585, 6635, 6636, 6641, 6643,
    6649, 6652, 6662, 6663, 6668, 6672, 6677, 6706, 6780, 6781, 6785, 6817, 6844, 6851, 6852, 6857,
    6864, 6879, 6884, 6910, 6961, 6970, 6972, 7006, 7086, 7089, 7091, 7096, 7117, 7141, 7159, 7188,
    7195, 7207, 7301, 7302, 7307, 7309, 7366, 7378, 7383, 7388, 7437, 7442, 7447, 7495, 7519, 7535,
    7536, 7539, 7541, 7612, 7648, 7660, 7693, 7696, 7699, 7710, 7714, 7733, 7734, 7739, 7750, 7797,
    7802, 7807, 7825, 7831, 7890, 7899, 7914, 7919, 7949, 7950, 7953, 7955, 8005, 8023, 8062, 8093,
    8094, 8099, 8111, 8112, 8117, 8120, 8121, 8126, 8131, 8167, 8215, 8290, 8293, 8317, 8318, 8319,
    8324, 8365, 8380, 8437, 8444, 8445, 8450, 8455, 8509, 8525, 8526, 8531, 8563, 8593, 8599, 8626,
    8628, 8664, 8704, 8737, 8759, 8760, 8765, 8768, 8769, 8774, 8848, 8876, 8877, 8878, 8882, 8889,
    8959, 8961, 8985, 8990,
]
sample = ds.select(FROZEN_INDICES).remove_columns("image").to_pandas()
sample["idx"] = FROZEN_INDICES
sample["question_id"] = [m["question_id"] for m in sample.metadata]
sample["anatomy"] = [m["location"] for m in sample.metadata]
sample["group"] = [GROUPS[(kind, m["content_type"])] for kind, m in zip(sample.question_type, sample.metadata)]
sample["answer"] = sample.answer.str.strip()
closed = sample.question_type == "closed"
sample.loc[closed, "answer"] = sample.loc[closed, "answer"].str.lower()
manifest = sample[["idx", "question_id", "group"]].to_csv(index=False, lineterminator="\n")
assert hashlib.sha256(manifest.encode()).hexdigest() == "0a80986f837c2e594ce54993cbb470130674a4624a8657fb64b4cedcdd335f35"
assert sample.idx.is_unique and (sample.groupby("group").size() == 100).all()
assert (sample[closed].groupby(["group", "answer"]).size() == 50).all()
display(sample.groupby("group").size())

## Original prompts, answer parsing and image conditions

In [ ]:
%%ministral
LETTERS = "ABCD"

def build_prompt(row):
    if row["question_type"] == "multiple_choice":
        opts = "\n".join(f"{L}. {c}" for L, c in zip(LETTERS, row["choices"]))
        return ("You are a medical assistant. Look at this image and answer the following multiple-choice "
                "question with ONLY the letter of the correct option (e.g. 'B'), nothing else.\n\n"
                f"Question: {row['question']}\nOptions:\n{opts}")
    return ("You are a medical assistant. Look at this image and answer the following question "
            "with ONLY the single word 'Yes' or 'No'.\n\n"
            f"Question: {row['question']}")

def parse_mcq(output):
    m = re.search(r"\b([A-D])\b", output)          # first standalone option letter
    return m.group(1) if m else "unparsed"

def parse_yesno(output):
    m = re.search(r"\b(yes|no)\b", output.lower())  # whichever appears first
    return m.group(1) if m else "unparsed"

def parse(row, output):
    m = re.search(r"<answer>(.*?)</answer>", output, re.S | re.I)   # MedVLM-R1 may answer as <answer>X</answer>
    text = m.group(1) if m else output
    return parse_mcq(text) if row["question_type"] == "multiple_choice" else parse_yesno(text)


ITEMS = [
    dict(idx=int(row["idx"]), question_id=row["question_id"], group=row["group"], anatomy=row["anatomy"],
         question=row["question"], true_answer=row["answer"], question_type=row["question_type"],
         prompt=build_prompt(row))
    for row in sample.to_dict("records")
]

def prediction(item, raw):
    return parse(item, raw)

def item_image(item, condition):
    if condition == "real":
        return ds[item["idx"]]["image"].convert("RGB")
    if condition == "white":
        return Image.new("RGB", (512, 512), condition)
    raise ValueError(f"Unknown image condition: {condition}")

## Load a pinned checkpoint and generate a direct answer

In [ ]:
%%ministral
class NumericsGuard(LogitsProcessor):
    def __init__(self):
        self.invalid = False

    def __call__(self, input_ids, scores):
        self.invalid |= bool((torch.isnan(scores) | torch.isposinf(scores)).any()
                             or torch.isneginf(scores).all(dim=-1).any())
        return scores


def load_model(tag):
    repo, revision = MODEL_SPECS[tag]
    config_files = ["config.json", "generation_config.json", "processor_config.json",
                    "preprocessor_config.json", "tokenizer.json", "tokenizer_config.json",
                    "special_tokens_map.json", "chat_template.jinja"]
    weights = (["consolidated.safetensors"] if tag == "ministral_3b_medvl"
               else ["model-*.safetensors", "model.safetensors.index.json"])
    print(f"Loading {repo} at {revision}", flush=True)
    snapshot = Path(snapshot_download(repo, revision=revision, token=HF_TOKEN,
                                      allow_patterns=config_files + weights))
    processor = AutoProcessor.from_pretrained(snapshot, local_files_only=True)
    # OpenMed uses Transformers tensor names inside a nonstandard filename.
    # Expose the conventional name without copying weights or changing the Hub cache.
    with TemporaryDirectory(prefix="ministral-checkpoint-") as local:
        checkpoint = snapshot
        if tag == "ministral_3b_medvl":
            checkpoint = Path(local)
            for filename in ("config.json", "generation_config.json"):
                (checkpoint / filename).symlink_to(snapshot / filename)
            (checkpoint / "model.safetensors").symlink_to(snapshot / "consolidated.safetensors")
        model, report = Mistral3ForConditionalGeneration.from_pretrained(
            checkpoint, local_files_only=True, use_safetensors=True, dtype=DTYPE,
            device_map={"": "cuda:0"}, attn_implementation="sdpa", output_loading_info=True,
        )
    faults = {k: report[k] for k in ("missing_keys", "unexpected_keys", "mismatched_keys", "error_msgs")
              if report.get(k)}
    if faults:
        raise RuntimeError(f"Incomplete weight load for {repo}: {faults}")
    # The study sets a new-token budget for every call.
    model.generation_config.max_length = None
    print(f"Loaded {repo} on {torch.cuda.get_device_name(0)} ({DTYPE})", flush=True)
    return model.eval(), processor


@torch.inference_mode()
def generate(model, processor, image, prompt):
    # An explicit empty system message suppresses the differing default system instructions.
    messages = [
        {"role": "system", "content": [{"type": "text", "text": ""}]},
        {"role": "user", "content": [{"type": "image", "image": image},
                                     {"type": "text", "text": prompt}]},
    ]
    inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                           return_dict=True, return_tensors="pt")
    if "pixel_values" not in inputs or not (inputs["input_ids"] == model.config.image_token_index).any():
        raise RuntimeError("The processor did not encode the image and its placeholder.")
    inputs = inputs.to(model.device, dtype=DTYPE)
    guard = NumericsGuard()
    output = model.generate(**inputs, do_sample=False, num_beams=1, use_cache=True,
                            max_new_tokens=MAX_NEW_TOKENS, logits_processor=LogitsProcessorList([guard]))
    tokens = output[0, inputs["input_ids"].shape[1]:].tolist()
    raw = processor.tokenizer.decode(tokens, skip_special_tokens=True, clean_up_tokenization_spaces=False)
    return raw, len(tokens), guard.invalid

## Save predictions and score each group

In [ ]:
%%ministral
SAMPLE_COLUMNS = ["idx", "question_id", "group"]
RESULT_COLUMNS = [*SAMPLE_COLUMNS, "anatomy", "question", "true_answer", "predicted_answer", "raw_output"]
SCORE_COLUMNS = ["group", "correct", "n", "accuracy"]

def write_csv(path, records, fields):
    temp = path.with_suffix(".tmp")
    with temp.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(records)
    temp.replace(path)

def scores(records):
    groups = sorted({r["group"] for r in records})
    output = []
    for group in groups:
        answers = [r for r in records if r["group"] == group]
        correct = sum(r["true_answer"] == r["predicted_answer"] for r in answers)
        output.append(dict(group=group, correct=correct, n=len(answers), accuracy=correct / len(answers)))
    return output

def run_items(tag, condition, folder, prefix, model, processor):
    folder.mkdir(parents=True, exist_ok=True)
    write_csv(folder / f"{prefix}_sample.csv", ITEMS, SAMPLE_COLUMNS)
    path = folder / f"{prefix}_results_{tag}.csv"
    rows = []
    if path.exists():
        with path.open(encoding="utf-8", newline="") as stream:
            reader = csv.DictReader(stream)
            if reader.fieldnames != RESULT_COLUMNS:
                raise ValueError(f"Unexpected columns in {path}")
            rows = list(reader)
        keys = RESULT_COLUMNS[:-2]
        if len(rows) > len(ITEMS) or any(
            any(row[k] != str(item[k]) for k in keys) for row, item in zip(rows, ITEMS)
        ):
            raise ValueError(f"Saved results do not match the frozen sample: {path}")
    for item in tqdm(ITEMS[len(rows):], desc=f"{tag} {condition}"):
        image = item_image(item, condition)
        raw, _, invalid = generate(model, processor, image, item["prompt"])
        pred = "unparsed" if invalid else prediction(item, raw)
        rows.append({**item, "predicted_answer": pred, "raw_output": raw})
        write_csv(path, rows, RESULT_COLUMNS)
        if len(rows) == 1 or len(rows) % 25 == 0:
            print(f"{tag} {condition}: saved {len(rows)}/{len(ITEMS)} predictions", flush=True)
    summary = scores(rows)
    write_csv(folder / f"{prefix}_scores_{tag}.csv", summary, SCORE_COLUMNS)
    print(tag, condition, sum(r["correct"] for r in summary), "/", len(rows),
          "correct;", sum(r["predicted_answer"] == "unparsed" for r in rows), "unparsed")
    display(pd.DataFrame(summary))
    return rows

In [ ]:
%%ministral
if not torch.cuda.is_available():
    raise RuntimeError("Select a CUDA GPU runtime before starting the run.")
settings = {
    "protocol": 2, "conditions": list(CONDITIONS), "dataset_revision": DATASET_REVISION, "models": MODEL_SPECS, "seed": SEED, "dtype": str(DTYPE),
    "max_new_tokens": MAX_NEW_TOKENS, "attention": "sdpa", "image_processor": "native Pixtral",
    "system_message": "", "do_sample": False, "num_beams": 1,
    "runtime": {p: version(p) for p in ("transformers", "accelerate", "tokenizers", "pillow", "torchvision")},
    "torch": str(torch.__version__), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "sample_and_prompts_sha256": hashlib.sha256(json.dumps(ITEMS, sort_keys=True).encode()).hexdigest(),
}
settings_text = json.dumps(settings, indent=2, sort_keys=True)
if PREVIOUS_OUTPUT and not OUTPUT_ROOT.exists():
    from tempfile import TemporaryDirectory
    previous = Path(PREVIOUS_OUTPUT)
    if (previous / "run_settings.json").read_text() != settings_text:
        raise ValueError(f"Previous run settings differ: {previous}")
    with TemporaryDirectory(dir=OUTPUT_ROOT.parent) as staging:
        restored = Path(staging) / "results"
        shutil.copytree(previous, restored)
        restored.rename(OUTPUT_ROOT)
    print(f"Restored previous predictions from {previous}", flush=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
record = OUTPUT_ROOT / "run_settings.json"
if record.exists() and record.read_text() != settings_text:
    raise ValueError("Run settings changed. Select a new output directory for this run.")
record.write_text(settings_text)

In [ ]:
%%ministral
def wilson(k, n, z=1.96):
    if n == 0: return (0, 0)
    p = k / n; d = 1 + z*z/n
    c = (p + z*z/(2*n)) / d
    h = z * math.sqrt(p*(1-p)/n + z*z/(4*n*n)) / d
    return c - h, c + h


def report(rows):
    results = pd.DataFrame(rows)
    results["correct"] = results.true_answer == results.predicted_answer
    n, k = len(results), int(results.correct.sum())
    print("Overall accuracy:", k / n, "95% Wilson CI:", wilson(k, n))
    display(results.groupby("anatomy").correct.agg(["count", "mean"]))
    yn = results[results.group.str.endswith("yesno")]
    labels = ["yes", "no", "unparsed"]
    print(classification_report(yn.true_answer, yn.predicted_answer, labels=labels[:2], zero_division=0))
    display(pd.DataFrame(confusion_matrix(yn.true_answer, yn.predicted_answer, labels=labels),
                         index=labels, columns=labels))
    display(results.loc[~results.correct, ["group", "question", "true_answer", "predicted_answer", "raw_output"]].head(30))

for tag in RUN_MODELS:
    model, processor = load_model(tag)
    try:
        for condition in CONDITIONS:
            folder = OUTPUT_ROOT / ("real" if condition == "real" else f"blank_control_{condition}")
            report(run_items(tag, condition, folder, "vqa", model, processor))
    finally:
        del model, processor
        gc.collect()
        torch.cuda.empty_cache()

In [ ]:
%%ministral
archive = shutil.make_archive(str(OUTPUT_ROOT), "zip", OUTPUT_ROOT)
print("Result bundle:", archive)
from IPython.display import FileLink, display
display(FileLink(os.path.relpath(archive)))

In [ ]:
ministral_runtime.close()